# Antardrishti — change detection, scored

**Question.** The shipped radar rule (mean(VV,VH) below −20 dB, IoU 0.489) misses 43% of flood water. Notebook 04 showed why: 28.3% of the missed water is brighter than −10 dB, and the missed-pixel median sits 9 dB above the caught one — a different population, not a mistuned threshold. Change detection asks each pixel a different question: *is it much darker than its own dry self?* This notebook measures whether that recovers the missing water, and what it costs.

**The rule** (`evaluation/change.py`, mirrored exactly in `backend/detection/sar.py`):

    water = post < −20 dB
            OR (post − pre < −3 dB  AND  post < −15 dB)

where `pre` is the median of a dry window before the flood, from the same relative orbit.

**What would count as an answer.** An IoU against the shipped rule on the same pixels; the missed water recovered, split into notebook 04's bins; and the false positives it adds. Change detection only replaces the darkness rule if IoU rises — recall bought with more precision than it is worth is not an improvement.

**Read before believing any number below.**

1. **The baseline is ours, not the dataset's.** Sen1Floods11 has no pre-flood radar. Each baseline is fetched from Earth Engine by `scripts/fetch_pre_event.py`: the 60 days ending 15 days before the flood image. If that window was itself wet — an early monsoon, a second flood — the pixel does not look darker, and change detection undercounts. The manifest records every window.
2. **Flooded vegetation stays out of the rule, and probably out of the labels.** Labels were drawn from optical imagery, which cannot see water under a canopy. A brightening branch for double bounce would mostly score as false positives here, so it is not tested.
3. **Processing mismatch, the same one the 0.489 already carries.** `S1Hand` chips are single, unfiltered scenes; production uses a speckle-filtered minimum composite. Relative comparisons between rules are sound; absolute numbers transfer approximately.


In [1]:
# One-time install. rasterio ships Windows wheels, so there is no compiler step.
!pip install rasterio numpy


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Fetch the baselines — once, and it takes a while

From `backend/`, in a terminal with Earth Engine authenticated:

```
python -m scripts.fetch_pre_event --sen1floods11 C:\sen1floods11
```

A few seconds per chip, 446 chips. It is **resumable**: interrupt it and run it again, and finished chips are skipped. Results land in `C:\sen1floods11\S1Pre\`, with `manifest.jsonl` recording the window, orbit and scene count for every chip — and the reason for every one that failed.


In [2]:
LOCAL = r"C:\sen1floods11"
PRE = None   # defaults to LOCAL\S1Pre, where the fetch script writes

import json
from collections import Counter
from pathlib import Path

root = Path(LOCAL)
pre_dir = Path(PRE) if PRE else root / "S1Pre"
if not root.exists():
    raise SystemExit(f"{root} does not exist")
if not pre_dir.exists():
    raise SystemExit(f"{pre_dir} does not exist - run scripts/fetch_pre_event.py first")

s1 = {p.name.replace("_S1Hand.tif", ""): p for p in root.rglob("*_S1Hand.tif")}
labels = {p.name.replace("_LabelHand.tif", ""): p for p in root.rglob("*_LabelHand.tif")}
pre = {p.name.replace("_S1Pre.tif", ""): p for p in pre_dir.glob("*_S1Pre.tif")}

keys = sorted(set(s1) & set(labels) & set(pre))
print(f"chips: {len(s1)}   with a baseline: {len(keys)}   without: {len(set(s1) & set(labels)) - len(keys)}")
if not keys:
    # An empty S1Pre folder passes the exists() check above. An early version
    # of the fetch script left one behind when it crashed, and the notebook
    # then ran on to a KeyError three cells later instead of stopping here.
    raise SystemExit(
        f"{pre_dir} has no *_S1Pre.tif files yet. Run the fetch first, from backend/:\n"
        f"  python -m scripts.fetch_pre_event --sen1floods11 {LOCAL} --show-dates\n"
        "check the dates it prints, then run it again without --show-dates."
    )

manifest = pre_dir / "manifest.jsonl"
if manifest.exists():
    records = [json.loads(line) for line in manifest.read_text(encoding="utf-8").splitlines() if line.strip()]
    failures = Counter(r.get("reason", "")[:70] for r in records if r.get("status") != "ok")
    if failures:
        print("\nwhy baselines are missing:")
        for reason, n in failures.most_common():
            print(f"  {n:>4}  {reason}")

print("\nEvery number below is over the chips WITH a baseline. If that is far")
print("fewer than 446, say so wherever the numbers are quoted.")

chips: 446   with a baseline: 434   without: 12

why baselines are missing:
    12  no Sentinel-1 scenes from orbit 95 between 2018-07-08 and 2018-09-06

Every number below is over the chips WITH a baseline. If that is far
fewer than 446, say so wherever the numbers are quoted.


In [3]:
import sys
EVALUATION = Path.cwd().resolve().parent.parent / "evaluation"
if not EVALUATION.exists():
    raise SystemExit(f"{EVALUATION} not found - start Jupyter from backend/notebooks/")
sys.path.insert(0, str(EVALUATION))

import numpy as np
import rasterio

import change as C
from metrics import evaluate, aggregate, format_table

print(f"rule: post < {C.DARK_DB}  OR  (post - pre < {C.DROP_DB} AND post < {C.CEILING_DB})")

rule: post < -20.0  OR  (post - pre < -5.0 AND post < -16.0)


In [4]:
# Per chip: post and pre as fused mean(VV,VH) dB, and the labels restricted to
# pixels the post image covers. Pixels with no baseline stay in: the change rule
# falls back to darkness there, so both rules are scored on exactly the same set.
LIMIT = None

def read_pair(path):
    with rasterio.open(path) as src:
        vv = src.read(1).astype(np.float64)
        vh = src.read(2).astype(np.float64)
    vv[vv < C.NODATA_FLOOR_DB] = np.nan
    vh[vh < C.NODATA_FLOOR_DB] = np.nan
    return C.fused(vv, vh)

loaded = []
for key in (keys[:LIMIT] if LIMIT else keys):
    try:
        post = read_pair(s1[key])
        before = read_pair(pre[key])
        with rasterio.open(labels[key]) as src:
            truth = src.read(1).astype(np.int8)
    except Exception as exc:
        print(f"skip {key}: {exc}")
        continue
    if before.shape != post.shape:
        print(f"skip {key}: baseline grid {before.shape} != chip grid {post.shape}")
        continue
    truth = truth.copy()
    truth[~np.isfinite(post)] = -1          # scored only where the post image has data
    if (truth != -1).any():
        loaded.append({"key": key, "post": post.astype(np.float32),
                       "pre": before.astype(np.float32), "truth": truth})

if not loaded:
    raise SystemExit("no chip could be loaded with a baseline - see the skip messages above")

common = sum(int((c["truth"] != -1).sum()) for c in loaded)
no_history = sum(int(((c["truth"] != -1) & ~np.isfinite(c["pre"])).sum()) for c in loaded)
print(f"{len(loaded)} chips, {common:,} scored pixels, "
      f"{no_history:,} ({no_history / max(common, 1):.1%}) without a baseline value")

428 chips, 97,993,540 scored pixels, 6,206 (0.0%) without a baseline value


## 2. Sweep the two new parameters

The drop and the ceiling are physically motivated starting points, not measurements. Sweep both; the validation block at the end still describes the constants **that ship**, because a score only applies to the rule it was measured on.

In [5]:
def score(rule):
    return aggregate([evaluate(rule(c["post"], c["pre"])[0], c["truth"]) for c in loaded])

drops = [-1.5, -2.0, -2.5, -3.0, -4.0, -5.0, -6.0]
ceilings = [-13.0, -14.0, -15.0, -16.0, -17.0]

grid = {}
for drop in drops:
    for ceiling in ceilings:
        grid[(drop, ceiling)] = score(C.change_rule(C.DARK_DB, drop, ceiling))["micro"]

print("IoU      " + "".join(f"{c:>8.0f}" for c in ceilings) + "   <- ceiling dB")
for drop in drops:
    row = "".join(f"{grid[(drop, c)]['iou']:>8.3f}" for c in ceilings)
    print(f"drop {drop:>4}{row}")

BEST = max(grid, key=lambda k: grid[k]["iou"])
print(f"\nbest: drop {BEST[0]} dB, ceiling {BEST[1]} dB, IoU {grid[BEST]['iou']:.3f}")
print(f"ships: drop {C.DROP_DB} dB, ceiling {C.CEILING_DB} dB, "
      f"IoU {grid[(C.DROP_DB, C.CEILING_DB)]['iou']:.3f}")

IoU           -13     -14     -15     -16     -17   <- ceiling dB
drop -1.5   0.325   0.358   0.398   0.432   0.458
drop -2.0   0.364   0.390   0.421   0.449   0.469
drop -2.5   0.404   0.422   0.444   0.466   0.479
drop -3.0   0.440   0.451   0.466   0.480   0.489
drop -4.0   0.488   0.490   0.495   0.499   0.501
drop -5.0   0.503   0.503   0.504   0.505   0.505
drop -6.0   0.502   0.502   0.502   0.502   0.501

best: drop -5.0 dB, ceiling -16.0 dB, IoU 0.505
ships: drop -5.0 dB, ceiling -16.0 dB, IoU 0.505


## 3. Head to head, on the same pixels

In [6]:
RULES = {
    f"darkness < {C.DARK_DB} dB (ships today)": C.darkness_rule(),
    f"change, drop {C.DROP_DB} / ceiling {C.CEILING_DB} (as coded)": C.change_rule(),
    f"change, drop {BEST[0]} / ceiling {BEST[1]} (sweep best)": C.change_rule(C.DARK_DB, *BEST),
}
results = {name: score(rule) for name, rule in RULES.items()}
print(format_table(results, "Sen1Floods11, chips with a baseline"))

base = list(results.values())[0]["micro"]
print(f"\n{'rule':<52}{'d IoU':>8}{'d recall':>10}{'d prec':>9}")
for name, agg in results.items():
    m = agg["micro"]
    print(f"{name:<52}{m['iou'] - base['iou']:>+8.3f}{m['recall'] - base['recall']:>+10.3f}"
          f"{m['precision'] - base['precision']:>+9.3f}")

Sen1Floods11, chips with a baseline
method                           IoU      F1    prec  recall   chips
--------------------------------------------------------------------
darkness < -20.0 dB (ships today)   0.485   0.653   0.752   0.577     428
change, drop -5.0 / ceiling -16.0 (as coded)   0.505   0.671   0.728   0.622     428
change, drop -5.0 / ceiling -16.0 (sweep best)   0.505   0.671   0.728   0.622     428

(micro-averaged: all pixels pooled)

rule                                                   d IoU  d recall   d prec
darkness < -20.0 dB (ships today)                     +0.000    +0.000   +0.000
change, drop -5.0 / ceiling -16.0 (as coded)          +0.020    +0.045   -0.024
change, drop -5.0 / ceiling -16.0 (sweep best)        +0.020    +0.045   -0.024


## 4. Which missed water does it recover — and what does it cost?

Notebook 04's bins, so this lines up with the question it was built to answer. The **bright** row is the headline: that is water no threshold could reach.

In [7]:
def recovery_table(rule):
    shipped = C.darkness_rule()
    totals = {b: {"missed": 0, "recovered": 0} for b in ("marginal", "moderate", "bright")}
    added_fp = 0
    for c in loaded:
        base_pred, _ = shipped(c["post"], c["pre"])
        change_pred, _ = rule(c["post"], c["pre"])
        for b, v in C.recovery(c["post"], c["truth"], base_pred, change_pred).items():
            totals[b]["missed"] += v["missed"]
            totals[b]["recovered"] += v["recovered"]
        added_fp += C.added_false_positives(c["truth"], base_pred, change_pred)
    return totals, added_fp

recovery, added_fp = recovery_table(C.change_rule())
print(f"{'bin':<10}{'missed':>12}{'recovered':>12}{'share':>9}")
print("-" * 43)
for b, v in recovery.items():
    share = v["recovered"] / v["missed"] if v["missed"] else 0.0
    print(f"{b:<10}{v['missed']:>12,}{v['recovered']:>12,}{share:>9.1%}")
recovered = sum(v["recovered"] for v in recovery.values())
print(f"\nwater recovered: {recovered:,}   dry pixels newly called water: {added_fp:,}")
# Two different questions, and they can disagree.
#  - Total mistakes: each recovered pixel removes one, each new false alarm adds one.
#  - IoU: a recovered pixel turns a miss into a hit WITHOUT enlarging the
#    denominator, while a false alarm enlarges it. So IoU rises whenever
#    recovered / added is greater than the current IoU - even when the rule
#    makes more mistakes in total.
print(f"total mistakes change: {added_fp - recovered:+,} pixels")
print(f"recovered / added = {recovered / max(added_fp, 1):.2f}; IoU rises if this exceeds the current IoU")

bin             missed   recovered    share
-------------------------------------------
marginal     1,654,957     392,071    23.7%
moderate     2,367,131      46,884     2.0%
bright          85,802           0     0.0%

water recovered: 438,955   dry pixels newly called water: 410,371
If the second number is larger than the first, the rule adds more error than it removes.


In [8]:
as_coded = results[list(RULES)[1]]["micro"]
validation_block = {
    "dataset": f"Sen1Floods11 HandLabeled ({len(loaded)} chips with a fetched pre-event baseline)",
    "iou": round(as_coded["iou"], 3),
    "precision": round(as_coded["precision"], 3),
    "recall": round(as_coded["recall"], 3),
}

payload = {
    "question": "Does change detection recover the water the darkness rule misses?",
    "chips_scored": len(loaded),
    "scored_pixels": common,
    "pixels_without_baseline": no_history,
    "rule": {"dark_db": C.DARK_DB, "drop_db": C.DROP_DB, "ceiling_db": C.CEILING_DB},
    "sweep_iou": {f"{d}|{c}": grid[(d, c)] for d, c in grid},
    "best": {"drop_db": BEST[0], "ceiling_db": BEST[1], **grid[BEST]},
    "head_to_head": {n: {"micro": a["micro"], "macro": a["macro"]} for n, a in results.items()},
    "recovery_by_bin": recovery,
    "added_false_positives": added_fp,
    "validation_block_for_backend": validation_block,
    "caveats": [
        "Baselines fetched from Earth Engine, 60 days ending 15 days before each flood image; a wet baseline window makes change detection undercount.",
        "Flooded vegetation is not in the rule and is probably missing from the optically drawn labels.",
        "S1Hand chips are single unfiltered scenes; production uses a speckle-filtered minimum composite.",
    ],
}
out = EVALUATION / "change_results.json"
out.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print(f"written to {out}\n")
print("If IoU rose, paste this into backend/detection/sar.py as CHANGE_VALIDATION:\n")
print(json.dumps(validation_block, indent=4))

written to C:\Users\punit\OneDrive\Desktop\Group_project\application\antardrishti\evaluation\change_results.json

If IoU rose, paste this into backend/detection/sar.py as CHANGE_VALIDATION:

{
    "dataset": "Sen1Floods11 HandLabeled (428 chips with a fetched pre-event baseline)",
    "iou": 0.505,
    "precision": 0.728,
    "recall": 0.622
}


## What to do with the numbers

**If IoU rose over the darkness rule.** Paste the block into `CHANGE_VALIDATION` in `backend/detection/sar.py`. Confidence, the accuracy note, the map styling and the PDF all read it from there, and the "not yet scored" note disappears by itself. Change detection is still opt-in (`"method": "change"` with a baseline window); making it the default is a separate decision, because it needs a baseline window on every request.

**If the sweep's best constants beat the coded ones clearly.** Change `CHANGE_DROP_DB` / `CHANGE_CEILING_DB` in `sar.py` *and* `DROP_DB` / `CEILING_DB` in `evaluation/change.py`, rerun this notebook, and paste the new block. `test_the_scored_rule_is_the_rule_that_ships` fails until both are changed.

**If IoU fell.** That is a result. Say so in Chapter 7, with the recovery table: it shows *which* missed water change detection could not reach, which is the next question.

**Whatever happens, quote the chip count.** Every number here is over chips that got a baseline. If the fetch failed for many — no scenes from the same orbit in the window, typically — the manifest says which, and a reader needs to know.
